In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv(r"/workspaces/pandas_prac/Aug_7/messy_inventory.csv")

In [5]:
df.shape

(39, 18)

In [6]:
df.dtypes

SKU                    str
ProductName            str
Category               str
Weight                 str
ListPrice              str
DiscountPercent        str
Price                  str
StockQty             int64
WarehouseCode        int64
Supplier               str
RestockDate            str
IsActive               str
Rating             float64
OriginCountry          str
Notes                  str
ProductID              str
Color                  str
Size                   str
dtype: object

In [3]:
df.head(5)

,SKU,ProductName,Category,Weight,ListPrice,DiscountPercent,Price,StockQty,WarehouseCode,Supplier,RestockDate,IsActive,Rating,OriginCountry,Notes
0,SKU-1001-RED-L,Classic Cotton T-Shirt,Apparel,180g,$19.99,10%,$17.99,120,234,Fabrico,2023-06-15,Yes,4.5,USA,NaN
1,SKU-1001-BLU-M,Classic Coton T-Shirt,apparel,0.18kg,$19.99,10%,$17.99,85,234,Fabrico,2023-06-15,Yes,4.5,USA,NaN
2,SKU-1002-BLK-S,classic cotton tshirt,Apparel,180g,$19.99,0%,$19.99,60,234,Fabrico,2023-06,Yes,4.5,USA,NaN
3,SKU-1003-GRN-XL,Slim Fit Denim Jeans,Apparel,650g,$54.99,20%,$43.99,45,234,Denimworks,2023-07-01,Yes,4.2,USA,NaN
4,SKU-1004-BLK-M,Slim-Fit Denim Jeans,apparel,0.65kg,$54.99,20%,$43.99,30,234,Denimworks,2023-07,yes,4.2,USA,Name variant


## `SKU`: multiple capture groups in one regex

**Decision:** `SKU-1234-RED-L` has three distinct pieces of information glued together. `.str.extract()` with three parenthesized groups pulls all three out in a single pass, `(\d+)` for the numeric ID, `([A-Z]+)` for the color, `([A-Z]+)` for the size.

In [4]:
sku_split = df["SKU"].str.extract(r"SKU-(\d+)-([A-Z]+)-([A-Z]+)")
df["ProductID"] = sku_split[0]
df["Color"] = sku_split[1]
df["Size"] = sku_split[2]

df[["SKU", "ProductID", "Color", "Size"]]

,SKU,ProductID,Color,Size
0,SKU-1001-RED-L,1001,RED,L
1,SKU-1001-BLU-M,1001,BLU,M
2,SKU-1002-BLK-S,1002,BLK,S
3,SKU-1003-GRN-XL,1003,GRN,XL
4,SKU-1004-BLK-M,1004,BLK,M
5,SKU-1005-WHT-L,1005,WHT,L
6,SKU-1006-BLK-L,1006,BLK,L
7,SKU-1007-SLV-M,1007,SLV,M
8,SKU-1008-BLU-M,1008,BLU,M
9,SKU-1009-BLK-S,1009,BLK,S


In [10]:
df["ProductName"] = df["ProductName"].astype(str).str.strip().str.lower()
df["ProductName"]

0           classic cotton t-shirt
1            classic coton t-shirt
2            classic cotton tshirt
3             slim fit denim jeans
4             slim-fit denim jeans
5       wireless bluetooth earbuds
6       wireless bluetooth earbuds
7     stainless steel water bottle
8     stainless steel water bottle
9                 yoga mat premium
10                yoga mat premium
11                  leather wallet
12                  leather wallet
13               running shoes pro
14               running shoes pro
15              ceramic coffee mug
16              ceramic coffee mug
17          bluetooth speaker mini
18          bluetooth speaker mini
19                kids rain jacket
20                kids rain jacket
21         adjustable dumbbell set
22         adjustable dumbbell set
23              desk organizer set
24              desk organizer set
25             insulated lunch bag
26             insulated lunch bag
27        mens cotton socks 3-pack
28       men's cotto

In [12]:
df["Category"] = df["Category"].astype(str).str.strip().str.title()
df["Category"]

0         Apparel
1         Apparel
2         Apparel
3         Apparel
4         Apparel
5     Electronics
6     Electronics
7            Home
8            Home
9         Fitness
10        Fitness
11    Accessories
12    Accessories
13       Footwear
14       Footwear
15           Home
16           Home
17    Electronics
18    Electronics
19        Apparel
20        Apparel
21        Fitness
22        Fitness
23           Home
24           Home
25           Home
26           Home
27        Apparel
28        Apparel
29    Electronics
30    Electronics
31        Outdoor
32        Outdoor
33           Home
34           Home
35        Apparel
36        Apparel
37        Outdoor
38        Outdoor
Name: Category, dtype: str

## Weight Normalization to Kg.

**Issue Identified:**  
The weight data contains mixed measurement units (`kg` and `gm`/`g`), preventing direct quantitative analysis and numerical aggregation.

**Fix Applied:**  
1. **Parsed Numerical Values & Units:** Extracted numeric values alongside their respective unit indicators.
2. **Unit Conversion:** Converted all gram values (`g/gm`) to kilogram (`kg`) by dividing by $1000$.
3. **Type Conversion:** Formatted the final result into a uniform numeric column represented purely in kg (`kg`).

In [21]:
def clean_weight(val):
    val = str(val).strip().lower()
    if val in ("nan", "n", ""):
        return np.nan
    num_match = re.search(r"([\d.]+)", val)
    if not num_match:
        return np.nan
    num = float(num_match.group(0))
    if "kg" in val:
        return f"{num} kg"
    elif "g" in val:
        return f"{round(num/1000,2)} kg"
    return val

df["Weight"] = df["Weight"].apply(clean_weight)
df["Weight"] = df["Weight"].astype(str).str.strip()
df["Weight"]

0      0.18 kg
1      0.18 kg
2      0.18 kg
3      0.65 kg
4      0.65 kg
5      0.06 kg
6     0.055 kg
7       0.4 kg
8       0.4 kg
9       1.2 kg
10      1.2 kg
11     0.09 kg
12     0.09 kg
13     0.85 kg
14     0.85 kg
15     0.32 kg
16     0.32 kg
17     0.21 kg
18     0.21 kg
19      0.3 kg
20      0.3 kg
21     15.0 kg
22     15.0 kg
23     0.55 kg
24     0.55 kg
25     0.25 kg
26     0.25 kg
27     0.12 kg
28     0.12 kg
29      0.1 kg
30    0.095 kg
31      3.2 kg
32      3.2 kg
33      0.9 kg
34      0.9 kg
35     0.08 kg
36     0.08 kg
37      1.1 kg
38      1.1 kg
Name: Weight, dtype: str